# NB00 — Descripción del pipeline experimental

**TFM · Máster Universitario en Inteligencia Artificial · VIU 2025-2026 · Víctor Rodríguez Rodríguez**

---

## Propósito de este notebook

Este notebook recoge la formulación del problema, la estrategia metodológica, la lista de artefactos que produce el pipeline y el orden en el que se ejecutan los notebooks. Todo lo desarrollado se recoge en profundidad en la memoria y se ha verificado experimentalmente en los notebooks posteriores.

---

## Problema

**Clasificación binaria BI-RADS sobre mamografía de cribado**: diferenciar estudios con hallazgos sospechosos (BI-RADS 4 o 5, requieren pasar a fase diagnóstica) de estudios no sospechosos (BI-RADS 1, 2 o 3, continúan en cribado rutinario). Es la frontera de decisión clínica más establecida en programas de cribado.

Dos niveles de granularidad se evalúan en paralelo:

| Nivel | Muestras | Prevalencia | Decisión clínica |
|-------|----------|-------------|------------------|
| **Estudio** | 4.999 estudios (4.000 train / 1.000 test) | 9,62 % | "¿Este examen requiere revisión prioritaria?" |
| **Mama** | 9.998 mamas (7.998 / 2.000) | 4,94 % | "¿Qué mama concentra la sospecha?" |

Dataset: **VinDr-Mammo** (Nguyen et al., 2023) — cohorte vietnamita, acceso libre en PhysioNet. Cuatro vistas estándar por estudio (L-CC, L-MLO, R-CC, R-MLO) con anotación BI-RADS por vista y densidad mamaria por mama.

## Estrategia: reutilizar AsymMirai como extractor congelado

En lugar de entrenar una red desde cero con preentrenamiento ImageNet, se reutiliza el **backbone ResNet-18 de AsymMirai** (Donnelly et al., 2024), preentrenado sobre EMBED. AsymMirai fue diseñado para predicción de riesgo de cáncer a 1-5 años, tarea distinta a la implementada en este trabajo, pero sus representaciones capturan conocimiento denso del dominio mamográfico (texturas fibroglandulares, calcificaciones, patrones de densidad, morfología de lesiones). El objetivo del trabajo es evaluar si esas representaciones son transferibles a clasificación diagnóstica del examen actual.

El backbone se congela completamente (`requires_grad = False`, `model.eval()`). Sobre las representaciones extraídas se entrenan cabezas clasificadoras ligeras.

## Dos puntos de extracción

| Punto | Contenido | Qué aporta |
|-------|-----------|------------|
| **A** | Embeddings de las 4 vistas tras el backbone: `(4, 512, 52, 64)` por estudio | Solo el backbone preentrenado |
| **B** | Mapas de asimetría bilateral `\|stretch*L − stretch*R\|` para pares CC y MLO: `(2, 512, 52, 64)` | Backbone + la contribución específica de AsymMirai (cálculo bilateral con pesos de *stretch* aprendidos) |

Comparar A vs B responde experimentalmente si la innovación de AsymMirai (cálculo de asimetría bilateral entre mamas) aporta valor en la nueva tarea. Comparar A vs A+B responde si combinar ambos aporta señal complementaria.

## Dos estrategias de reducción espacial

Los mapas `(512, 52, 64)` son inviables de almacenar sin reducción para 5.000 estudios. Comparamos dos poolings sobre el mismo mapa:

| Estrategia | Operación | Dim. por mapa |
|-----------|-----------|---------------|
| **`gg` (GAP + GMP)** | Global Average Pooling + Global Max Pooling | 1024 |
| **`22` (pool adaptativo 2×2)** | AdaptiveAvg(2×2) + AdaptiveMax(2×2) | 4096 |

`gg` colapsa toda la dimensión espacial y `22` preserva un mapa 2×2 que podría capturar localización de lesiones.

## Diseño experimental: 10 configuraciones × 9 cabezas

Cruzando contenido del input × pooling × granularidad obtenemos 10 configuraciones base:

| Config | Granul. | Contenido | Pooling | Dim. input |
|--------|---------|-----------|---------|-----------:|
| `E_A_gg`, `E_A_22`   | Estudio | Punto A (4 vistas) | gg / 22 | 4096 / 16 384 |
| `E_B_gg`, `E_B_22`   | Estudio | Punto B (2 mapas asimetría) | gg / 22 | 2048 / 8 192 |
| `E_AB_gg`, `E_AB_22` | Estudio | A + B concatenados | gg / 22 | 6144 / 24 576 |
| `M_A_gg`, `M_A_22`   | Mama    | Punto A (2 vistas de esa mama) | gg / 22 | 2048 / 8 192 |
| `M_AB_gg`, `M_AB_22` | Mama    | A + B del par correspondiente | gg / 22 | 4096 / 16 384 |

Sobre cada configuración se entrenan hasta 9 cabezas clasificadoras:

- **Lineales regularizados**: `logreg_l1`, `logreg_l2`, `logreg_en` (ElasticNet).
- **Árboles clásicos**: `rf` (Random Forest), `extratrees` (Extra Trees).
- **Gradient boosting**: `histgb` (HistGradientBoosting), `xgb` (XGBoost), `lgbm` (LightGBM).
- **Neural**: `mlp` (perceptrón multicapa con BatchNorm + Dropout).

Todas comparten un tratamiento del desbalanceo equivalente (`class_weight='balanced'` en modelos lineales/árboles clásicos, `scale_pos_weight` o `sample_weight = n_neg/n_pos` en boosting, `pos_weight = n_neg/n_pos` en el MLP), con el fin de que las diferencias observadas reflejen las propiedades de cada cabeza.

ElasticNet se restringe a las 4 configuraciones a nivel mama por coste computacional. Total: 84 modelos base.

## Protocolo de validación cruzada

- Split externo: el predefinido de VinDr-Mammo (4.000 train / 1.000 test).
- Validación cruzada externa de 5 folds sobre el training pool:
    - `StratifiedKFold` a nivel estudio.
    - `StratifiedGroupKFold` con `groups=study_id` a nivel mama (evita que las dos mamas del mismo paciente caigan en folds distintos).
- Hold-out interno 80/20 dentro de cada fold para búsqueda de hiperparámetros. Tras seleccionar los mejores, se reentrena con el training completo del fold.
- Predicción sobre test: ensemble por promedio de los 5 modelos del fold.

## Extensiones sobre los modelos base

- **Fusión tardía con densidad mamaria** evaluada en tres variantes: fusión a nivel estudio, a nivel mama, y evaluación exhaustiva sobre las 36 combinaciones a nivel mama × 3 cabezas de fusión = 108 modelos.
- **Calibración post-hoc** (Platt scaling y regresión isotónica) sobre los dos mejores candidatos, seleccionando el método que preserva AUC (verificado con DeLong) y minimiza ECE.

## Métricas y estadística

- **Discriminación**: AUC con IC 95 % por bootstrap no paramétrico, Average Precision.
- **Calibración**: Brier score, Expected Calibration Error (ECE), reliability diagrams.
- **Comparaciones pareadas**: test de DeLong dentro del mismo nivel de granularidad.

## Modelos finales seleccionados

| Uso | Modelo | Métricas sobre test hold-out |
|-----|--------|------------------------------|
| **A nivel estudio** | `M_A_22 + XGBoost`, agregado por `max(L,R)`, calibrado con Platt | AUC = 0,689 [0,628-0,748] · AP = 0,363 · Brier = 0,080 · ECE = 0,019 |
| **A nivel mama** | `M_A_gg + MLP`, calibrado con Platt | AUC = 0,687 [0,629-0,742] · Brier = 0,044 · ECE = 0,008 |

## Estructura y orden de ejecución

```
notebooks/
├── 00_overview.ipynb                    - este notebook
├── 01_inspeccion_modelo.ipynb           - inspección de AsymMirai cargado
├── 02_extraccion_un_estudio.ipynb       - validación del pipeline sobre 1 estudio
├── 03_extraccion_masiva.ipynb           - extracción GAP+GMP de los 5000 estudios
├── 08_extraccion_pool22.ipynb           - re-extracción con pool adaptativo 2×2
├── 09_pipeline_unificado.ipynb          - núcleo experimental: 84 modelos base
├── 10_evaluacion_v2.ipynb               - métricas, DeLong, subgrupos, matrices
├── 11_fusion_densidad_v2.ipynb          - fusión con densidad a nivel estudio
├── 11b_fusion_densidad_mama.ipynb       - fusión con densidad a nivel mama
├── 11c_fusion_densidad_exhaustivo.ipynb - 108 modelos de fusión sobre nivel mama
├── 12_calibracion_posthoc.ipynb         - Platt / isotónica sobre los finales
├── 13_ejemplos_birads.ipynb             - Ilustración 2 de la memoria
└── 14_analisis_sonda_edad.ipynb         - ¿predice el modelo la edad implícitamente?

src/
├── tfm_pipeline.py    - builders, grids, hold-out interno, kfold unificado
└── tfm_eval.py        - bootstrap, DeLong, ECE, agregación Mama y Estudio
```

**Orden recomendado para reproducción completa**: `00 - 01 - 02 - 03 - 08 - 09 - 10 - 11 - 11b - 11c - 12 - 13 - 14`.

In [1]:
# Esta celda comprueba que las rutas de archivos y carpetas clave existen, y crea las carpetas de salida si no existen.
import os
from pathlib import Path

BASE = os.environ.get('TFM_PROJECT_ROOT', os.path.abspath(os.path.join(os.getcwd(), '..')))


ASYMMIRAI = os.path.join(BASE, 'AsymMirai')
DATA = os.path.join(BASE, 'Data', 'vindr-mammo')
NOTEBOOKS = os.path.join(BASE, 'notebooks')
OUTPUTS = os.path.join(BASE, 'outputs')
FEATURES_DIR = os.path.join(OUTPUTS, 'Features')
PRED_DIR = os.path.join(OUTPUTS, 'Predicciones')
MODELS_DIR = os.path.join(OUTPUTS, 'Models')
PLOTS_DIR = os.path.join(OUTPUTS, 'Plots')

for d in (FEATURES_DIR, PRED_DIR, MODELS_DIR, PLOTS_DIR):
    Path(d).mkdir(parents=True, exist_ok=True)

# Archivos clave
WEIGHTS = os.path.join(ASYMMIRAI, 'snapshots', 'trained_asymmirai.pt')
BREAST_CSV = os.path.join(DATA, 'breast-level_annotations.csv')
IMG_DIR = os.path.join(DATA, 'images')

SEED = 42

print('Verificación de rutas:')
for name, p in [('Pesos AsymMirai',   WEIGHTS), ('Annotations VinDr', BREAST_CSV), ('Carpeta imágenes',  IMG_DIR), ('Notebooks', NOTEBOOKS)]:
    sym = 'OK' if os.path.exists(p) else 'FALTA'
    print(f'{sym} {name}: {p}')

print('\nCarpetas de salida (creadas si no existían):')
for name, p in [('Features', FEATURES_DIR), ('Predicciones', PRED_DIR), ('Models', MODELS_DIR), ('Plots', PLOTS_DIR)]:
    print(f'OK {name}: {p}')


Verificación de rutas:
OK Pesos AsymMirai: c:\Users\victo\Documents\TFM\Proyecto\AsymMirai\snapshots\trained_asymmirai.pt
OK Annotations VinDr: c:\Users\victo\Documents\TFM\Proyecto\Data\vindr-mammo\breast-level_annotations.csv
OK Carpeta imágenes: c:\Users\victo\Documents\TFM\Proyecto\Data\vindr-mammo\images
OK Notebooks: c:\Users\victo\Documents\TFM\Proyecto\notebooks

Carpetas de salida (creadas si no existían):
OK Features: c:\Users\victo\Documents\TFM\Proyecto\outputs\Features
OK Predicciones: c:\Users\victo\Documents\TFM\Proyecto\outputs\Predicciones
OK Models: c:\Users\victo\Documents\TFM\Proyecto\outputs\Models
OK Plots: c:\Users\victo\Documents\TFM\Proyecto\outputs\Plots
